## 1. Dataset Loading and Inspection

In this section, we load the competition dataset and inspect the structure, columns, row counts, missing values, and basic characteristics of the three sources and ground truth.

In [ ]:
import pandas as pd

df = pd.read_csv("some_file.tsv", sep="\t")
df.head()

base = r"E:\Amazon-ML-Challenge-2026\data\student_resource"

train_source1 = pd.read_csv(f"{base}\\dataset\\train\\train_source1.tsv", sep="\t")
train_source2 = pd.read_csv(f"{base}\\dataset\\train\\train_source2.tsv", sep="\t")
train_source3 = pd.read_csv(f"{base}\\dataset\\train\\train_source3.tsv", sep="\t")
train_ground_truth = pd.read_csv(f"{base}\\dataset\\train\\train_ground_truth.tsv", sep="\t")

test_source1 = pd.read_csv(f"{base}\\dataset\\test\\test_source1.tsv", sep="\t")
test_source2 = pd.read_csv(f"{base}\\dataset\\test\\test_source2.tsv", sep="\t")
test_source3 = pd.read_csv(f"{base}\\dataset\\test\\test_source3.tsv", sep="\t")

print("Loaded OK")

Loaded OK


In [2]:
for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3),
    ("train_ground_truth", train_ground_truth),
    ("test_source1", test_source1),
    ("test_source2", test_source2),
    ("test_source3", test_source3),
]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("missing values:\n", df.isna().sum())


=== train_source1 ===
shape: (2206821, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

=== train_source2 ===
shape: (5034616, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

=== train_source3 ===
shape: (5285603, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

=== train_ground_truth ===
shape: (2206821, 2)
columns: ['source1_entity_id', 'matched_entity_ids']
missing values:
 source1_entity_id          0
matched_entity_ids    123247
dtype: int64

=== test_source1 ===
shape: (1732544, 4)
columns: ['entity_id', 'business_

## 2. Ground Truth and Dataset Distribution

Before designing the matching system, we inspect the distribution of
matches and countries in the training data.

In [3]:
train_source1.head(5)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [4]:
train_source2.head(5)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [5]:
train_ground_truth.head(10)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [6]:
# Confirm the missing ground truth rows really are "no match" (empty string vs NaN)
print(train_ground_truth['matched_entity_ids'].isna().sum())
print((train_ground_truth['matched_entity_ids'] == '').sum())

# How many matches does a typical Source 1 entity have? (only for non-missing rows)
match_counts = train_ground_truth['matched_entity_ids'].dropna().apply(lambda x: len(x.split(',')))
print(match_counts.describe())

# Country distribution across sources
print(train_source1['country'].value_counts())
print(train_source2['country'].value_counts())

123247
0
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64
country
US       1323633
India     883188
Name: count, dtype: int64
country
US       3016817
India    2017799
Name: count, dtype: int64


In [7]:
import re

def has_devanagari(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[\u0900-\u097F]', text))

for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3)
]:
    n = df['business_name'].apply(has_devanagari).sum()
    print(f"{name}: {n} / {len(df)} names contain Devanagari script ({n/len(df)*100:.2f}%)")

train_source1: 0 / 2206821 names contain Devanagari script (0.00%)
train_source2: 269424 / 5034616 names contain Devanagari script (5.35%)
train_source3: 158003 / 5285603 names contain Devanagari script (2.99%)


## 3. Train-Validation Split

The training data contains Source 1 records together with their ground-truth
matches from Source 2 and Source 3.

To evaluate our matching approach before applying it to the unseen test data,
we split the Source 1 entities into two groups:

- **80% training/development set** — used to develop the baseline approach.
- **20% validation set** — used to evaluate the approach on unseen Source 1 entities.

The split is random and uses a fixed random seed (`random_state=42`) so that
the same validation set can be reproduced across experiments.

Source 2 and Source 3 are not split at this stage because they serve as the
candidate databases against which Source 1 entities are matched.

In [10]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_source1['entity_id'],
    test_size=0.20,
    random_state=42
)

print("Training Source 1 entities:", len(train_ids))
print("Validation Source 1 entities:", len(val_ids))

# Sanity check: no overlap, and counts add up
assert len(set(train_ids) & set(val_ids)) == 0
assert len(train_ids) + len(val_ids) == len(train_source1)
print("Split check passed.")

Training Source 1 entities: 1765456
Validation Source 1 entities: 441365
Split check passed.


In [11]:
train_source1_split = train_source1[
    train_source1['entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_source1 = train_source1[
    train_source1['entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

train_ground_truth_split = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_ground_truth = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

print("Train S1:", train_source1_split.shape)
print("Validation S1:", val_source1.shape)
print("Train ground truth:", train_ground_truth_split.shape)
print("Validation ground truth:", val_ground_truth.shape)

Train S1: (1765456, 4)
Validation S1: (441365, 4)
Train ground truth: (1765456, 2)
Validation ground truth: (441365, 2)


## 4. Baseline Matching Approach

We now develop a simple entity-resolution baseline using the training data
and evaluate its predictions on the held-out validation set.

The baseline will first focus on high-confidence matching rules and
candidate generation before introducing more complex matching methods.

### 4.1 Validation Evaluation

The competition evaluates predictions using macro-averaged F₀.₅ across
Source 1 entities. Since precision is weighted more heavily than recall,
the evaluation function measures the overlap between the predicted and
ground-truth match sets for each Source 1 entity.

This function will be used to evaluate our baseline on the held-out
validation set before generating test predictions.

In [12]:
def parse_ids(cell):
    """Convert a comma-separated ID string (or NaN) into a set of IDs."""
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))


def f_beta_score(true_set, pred_set, beta=0.5):
    """Compute F_beta for one entity given its true and predicted match sets."""
    # Singleton case: no true matches
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0

    # We have true matches but predicted nothing
    if len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)

    if precision == 0 and recall == 0:
        return 0.0

    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


def evaluate_predictions(ground_truth_df, predictions_df):
    """
    ground_truth_df: columns [source1_entity_id, matched_entity_ids]
    predictions_df:  columns [source1_entity_id, matched_entity_ids]
    Returns macro-averaged F_0.5 across all entities in ground_truth_df.
    """
    gt = ground_truth_df.set_index('source1_entity_id')['matched_entity_ids']
    pred = predictions_df.set_index('source1_entity_id')['matched_entity_ids']

    scores = []
    for entity_id in gt.index:
        true_set = parse_ids(gt.loc[entity_id])
        # If we didn't predict anything for this entity at all, treat as empty
        pred_cell = pred.loc[entity_id] if entity_id in pred.index else None
        pred_set = parse_ids(pred_cell)
        scores.append(f_beta_score(true_set, pred_set))

    return sum(scores) / len(scores)

In [13]:
import pandas as pd

test_gt = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S3-00812']
})
test_pred = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S2-00193,S3-00812']
})

score = evaluate_predictions(test_gt, test_pred)
print(score)  # PDF says this should be 0.714

0.7142857142857142


## Evaluation Function — Verification

Before using the evaluation function on our real validation data, we tested it
with a small manually constructed example.

The ground truth contained two correct matches, while the prediction contained
those two matches plus one incorrect match. The resulting F_0.5 score was
approximately 0.7143, confirming that the evaluation function penalizes the
extra false match while giving credit for the two correct matches.

This is only a test of the evaluation code. No model has been trained and no
real validation score has been calculated yet.